In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import time
from smbus2 import SMBus,i2c_msg
from scservo_sdk import *                    # Uses SCServo SDK library
import MCP342x

# Control table address
ADDR_SCS_TORQUE_ENABLE     = 40
ADDR_SCS_GOAL_ACC          = 41
ADDR_SCS_GOAL_POSITION     = 42
ADDR_SCS_GOAL_SPEED        = 46
ADDR_SCS_PRESENT_POSITION  = 56
ADDR_SCS_MOVING_STATUS = 66

# Default setting
BAUDRATE                    = 115200           # SCServo default baudrate : 1000000
DEVICENAME                  = '/dev/ttyUSB0'    # Check which port is being used on your controller
                                                # ex) Windows: "COM1"   Linux: "/dev/ttyUSB0" Mac: "/dev/tty.usbserial-*"
# dmesg | grep tty
protocol_end                = 0           # SCServo bit end(STS/SMS=0, SCS=1)

In [2]:
# Initialize PortHandler instance
# Set the port path
# Get methods and members of PortHandlerLinux or PortHandlerWindows
portHandler = PortHandler(DEVICENAME)
portHandler.setPacketTimeoutMillis(100)
# Initialize PacketHandler instance
# Get methods and members of Protocol
packetHandler = PacketHandler(protocol_end)

# Open port
if portHandler.openPort():
    print("Succeeded to open the port")
else:
    print("Failed to open the port")
    print("Press any key to terminate...")
    getch()
    quit()

# Set port baudrate
if portHandler.setBaudRate(BAUDRATE):
    print("Succeeded to change the baudrate")
else:
    print("Failed to change the baudrate")
    print("Press any key to terminate...")
    getch()
    quit()

Succeeded to open the port
Succeeded to change the baudrate


In [3]:
# Close port
portHandler.closePort()

In [4]:
i2cbus = SMBus(1)
MCP3424_fiber=MCP342x.MCP342x(i2cbus, 0x68, device='MCP3424', channel=0, gain=1, resolution=12, continuous_mode=False, scale_factor=1.0, offset=0.0)
MCP3424_pinhole=MCP342x.MCP342x(i2cbus, 0x68, device='MCP3424', channel=1, gain=1, resolution=12, continuous_mode=False, scale_factor=1.0, offset=0.0)
MCP3424_ref=MCP342x.MCP342x(i2cbus, 0x68, device='MCP3424', channel=2, gain=1, resolution=12, continuous_mode=False, scale_factor=1.0, offset=0.0)

In [40]:
sts3032_dict={0:[2,'1x'], 1:[1,'1y'], 2:[4,'2x'], 3:[3,'2y'], 4:[4,'3x'], 5:[3,'3y'], 6:[2,'4x'],7:[1,'4y']} # dict {index:[ID, servo name]}
class sts3032:

    def __init__(self, channel, portHandler, packetHandler):
        self.SCS_ID=sts3032_dict[channel][0]
        self.turn_num = 0
        self.SCS_MOVING_SPEED = 1500          # SCServo moving speed
        self.SCS_MOVING_ACC   = 50          # SCServo moving acc
        self.raw_angle_current = 0
        #atexit.register(self.home)
        # Write SCServo acc
        scs_comm_result, scs_error = packetHandler.write1ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_GOAL_ACC, self.SCS_MOVING_ACC)
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print("%s" % packetHandler.getRxPacketError(scs_error))

        print('SCServo acc set!')
        # Write SCServo speed
        scs_comm_result, scs_error = packetHandler.write2ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_GOAL_SPEED, self.SCS_MOVING_SPEED)
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print("%s" % packetHandler.getRxPacketError(scs_error))
        print('SCServo speed set!')

    def set_zero(self):
        #Set Zeros
        scs_comm_result, scs_error = packetHandler.write1ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_TORQUE_ENABLE, 128)
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print("%s" % packetHandler.getRxPacketError(scs_error))
        self.turn_num = 0
        self.angle_current = 2048
        try:
            scs_present_position_speed, scs_comm_result, scs_error = packetHandler.read4ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_PRESENT_POSITION)
            if scs_comm_result != COMM_SUCCESS:
                print('result',packetHandler.getTxRxResult(scs_comm_result))
            elif scs_error != 0:
                print('error',packetHandler.getRxPacketError(scs_error))
            print('SCServo zero set!')
            return 0
        except:
            print('Read not sucessful!')
            return 1

    def set_speed(self, set_speed):
        # Write SCServo speed
        self.SCS_MOVING_SPEED=set_speed
        scs_comm_result, scs_error = packetHandler.write2ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_GOAL_SPEED, self.SCS_MOVING_SPEED)
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print("%s" % packetHandler.getRxPacketError(scs_error))
        print('SCServo speed set!')
        
    def set_angle(self, goal_position):

        if goal_position>=0:
            val=0b0000000000000000|abs(goal_position)
            scs_goal_position=val
        elif goal_position<0:
            val=0b1000000000000000|abs(goal_position)
            scs_goal_position=val

        #pre-read
        scs_present_position_speed, scs_comm_result, scs_error = packetHandler.read4ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_PRESENT_POSITION)
        if scs_comm_result != COMM_SUCCESS:
            print(packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print(packetHandler.getRxPacketError(scs_error))
        self.raw_angle_current = SCS_LOWORD(scs_present_position_speed)

        # Write SCServo goal position
        scs_comm_result, scs_error = packetHandler.write2ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_GOAL_POSITION, scs_goal_position)
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print("%s" % packetHandler.getRxPacketError(scs_error))

        i=0
        while i<5000:
            # Read SCServo present position 
            scs_present_position_speed, scs_comm_result, scs_error = packetHandler.read4ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_PRESENT_POSITION)
            if scs_comm_result != COMM_SUCCESS:
                print(packetHandler.getTxRxResult(scs_comm_result))
            elif scs_error != 0:
                print(packetHandler.getRxPacketError(scs_error))
            # Read SCServo present status
            scs_present_status, scs_comm_result, scs_error = packetHandler.read4ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_MOVING_STATUS)
            if scs_comm_result != COMM_SUCCESS:
                print(packetHandler.getTxRxResult(scs_comm_result))
            elif scs_error != 0:
                print(packetHandler.getRxPacketError(scs_error))
            scs_present_position = SCS_LOWORD(scs_present_position_speed)
            if abs(scs_present_position-self.raw_angle_current)>3500:
                self.turn_num=self.turn_num+int(np.sign(self.raw_angle_current-scs_present_position))
            self.raw_angle_current=scs_present_position
            self.angle_current=scs_present_position+4096*self.turn_num

            scs_present_status=scs_present_status&0x0001
            if i%10==0:
                print(i,scs_present_status,self.raw_angle_current, self.angle_current, goal_position)
            i=i+1

            if (abs(goal_position - self.angle_current) ==0) and (scs_present_status==0):
                print(i,scs_present_status,self.raw_angle_current, self.angle_current, goal_position)
                break
    
    def home(self):
        self.set_angle(2048)

    def torque_disable(self):
        scs_comm_result, scs_error = packetHandler.write1ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_TORQUE_ENABLE, 0)
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print("%s" % packetHandler.getRxPacketError(scs_error))

    def torque_enable(self):
        scs_comm_result, scs_error = packetHandler.write1ByteTxRx(portHandler, self.SCS_ID, ADDR_SCS_TORQUE_ENABLE, 1)
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))
        elif scs_error != 0:
            print("%s" % packetHandler.getRxPacketError(scs_error))

In [45]:
#A bigger Class that contains all the motors:
class servoset:
    def __init__(self, servo_list, portHandler, packetHandler):
        self.SCS_ID_list=[]
        for servo in servo_list:
            self.SCS_ID_list.append(servo.SCS_ID)
        #initialize turn numbers
        self.turn_num=np.zeros(len(self.SCS_ID_list))
    
    def set_zero(self):
        for servo in servo_list:
            iteration=1
            while 1:
                print('Set Zero Trail ',iteration)
                result=servo.set_zero()
                if result==0:
                    break
                iteration+=1
        #Set turn number to be zero when we set zero on all the motors.
        self.turn_num=np.zeros(len(self.SCS_ID_list))

    def home(self):
        goal_position_list=[]
        for i in range(len(self.SCS_ID_list)):
            goal_position_list.append(2048)
        self.set_angle(goal_position_list)

    def set_angle(self, goal_position_list):

        ADDR_STS_GOAL_ACC          = 41
        ADDR_STS_GOAL_POSITION     = 42
        ADDR_STS_GOAL_SPEED        = 46
        ADDR_STS_PRESENT_POSITION  = 56

        scs_goal_position=[]
        for goal_position in goal_position_list:
            if goal_position>=0:
                val=0b0000000000000000|abs(goal_position)
                scs_goal_position.append(val)
            elif goal_position<0:
                val=0b1000000000000000|abs(goal_position)
                scs_goal_position.append(val)

        SCS_MOVING_STATUS_THRESHOLD = 0               # SCServo moving status threshold
        protocol_end                = 0                 # SCServo bit end(STS/SMS=0, SCS=1)

        # Initialize GroupSyncWrite instance
        groupSyncWrite = GroupSyncWrite(portHandler, packetHandler, ADDR_STS_GOAL_POSITION, 2)

        # Initialize GroupSyncRead instace for Present Position
        groupSyncRead = GroupSyncRead(portHandler, packetHandler, ADDR_STS_PRESENT_POSITION, 4)

        # Add parameter storage for SCServo present position value
        for SCS_ID in self.SCS_ID_list:
            scs_addparam_result = groupSyncRead.addParam(SCS_ID)
            if scs_addparam_result != True:
                print("[ID:%03d] groupSyncRead addparam failed" % SCS_ID)
                quit()

        # Allocate goal position value into byte array
        # Add SCServo goal position values to the Syncwrite parameter storage
        index=0
        for SCS_ID in self.SCS_ID_list:
            param_goal_position = [SCS_LOBYTE(scs_goal_position[index]), SCS_HIBYTE(scs_goal_position[index])]
            scs_addparam_result = groupSyncWrite.addParam(SCS_ID, param_goal_position)
            index+=1
            if scs_addparam_result != True:
                print("[ID:%03d] groupSyncWrite addparam failed" % SCS_ID)
                quit()

        # Syncwrite goal position
        scs_comm_result = groupSyncWrite.txPacket()
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))

        # Clear syncwrite parameter storage
        groupSyncWrite.clearParam()

        #Pre-read
        scs_comm_result = groupSyncRead.txRxPacket()
        if scs_comm_result != COMM_SUCCESS:
            print("%s" % packetHandler.getTxRxResult(scs_comm_result))

        scs_present_position_speed = []

        for SCS_ID in self.SCS_ID_list:
            scs_getdata_result = groupSyncRead.isAvailable(SCS_ID, ADDR_STS_PRESENT_POSITION, 4)
            if scs_getdata_result == True:
                scs_present_position_speed.append(groupSyncRead.getData(SCS_ID, ADDR_STS_PRESENT_POSITION, 4))
            else:
                print("[ID:%03d] groupSyncRead getdata failed" % SCS_ID)
                scs_present_position_speed.append(0)

        status_string='Start Position: '+'\t'
        scs_present_position_list=[]

        index=0
        for SCS_ID in self.SCS_ID_list:
            scs_present_position = SCS_LOWORD(scs_present_position_speed[index])
            scs_present_position_list.append(scs_present_position)
            index+=1

        multi_position_list=[]
        for index in range(len(self.SCS_ID_list)): 
            multi_position_list.append(int(scs_present_position_list[index]+self.turn_num[index]*4096))
            status_string+='[ID:'+f'{self.SCS_ID_list[index]:03d}'+'] Goal:'+f'{goal_position_list[index]:03d}'+' Pres:'+f'{multi_position_list[index]:03d}'+'\t'

        scs_present_position_list_cache=scs_present_position_list.copy()
        print(status_string)

        iteration=0
        while iteration<2000:
            # Syncread present single turn position
            scs_comm_result = groupSyncRead.txRxPacket()
            if scs_comm_result != COMM_SUCCESS:
                print("%s" % packetHandler.getTxRxResult(scs_comm_result))

            scs_present_position_speed = []

            for SCS_ID in self.SCS_ID_list:
                # Check if groupsyncread data of SCServo is available
                scs_getdata_result = groupSyncRead.isAvailable(SCS_ID, ADDR_STS_PRESENT_POSITION, 4)
                if scs_getdata_result == True:
                    # Get SCServo#1 present position value
                    scs_present_position_speed.append(groupSyncRead.getData(SCS_ID, ADDR_STS_PRESENT_POSITION, 4))
                else:
                    print("[ID:%03d] groupSyncRead getdata failed" % SCS_ID)
                    scs_present_position_speed.append(0)
            
            status_string='Iteration: '+str(iteration)+'\t'
            # Printing current status
            scs_present_position_list=[]

            index=0
            for SCS_ID in self.SCS_ID_list:
                scs_present_position = SCS_LOWORD(scs_present_position_speed[index])
                scs_present_position_list.append(scs_present_position)
                index+=1

            multi_position_list=[]
            #determine whether turn number has been changed
            for index in range(len(self.SCS_ID_list)): 
                if abs(scs_present_position_list[index]-scs_present_position_list_cache[index])>3500:
                    if scs_present_position_list[index]-scs_present_position_list_cache[index]>0:
                        self.turn_num[index]=self.turn_num[index]-1
                    elif scs_present_position_list[index]-scs_present_position_list_cache[index]<0:
                        self.turn_num[index]=self.turn_num[index]+1

                multi_position_list.append(int(scs_present_position_list[index]+self.turn_num[index]*4096))

                status_string+='[ID:'+f'{self.SCS_ID_list[index]:03d}'+'] Goal:'+f'{goal_position_list[index]:03d}'+' Pres:'+f'{multi_position_list[index]:03d}'+'\t'

            scs_present_position_list_cache=scs_present_position_list.copy()

            if iteration%100==0:
                print(status_string)
            
            #count how many motors have finished moving
            is_done=0
            for i in range(len(self.SCS_ID_list)): 
                if abs(goal_position_list[i] - multi_position_list[i]) == SCS_MOVING_STATUS_THRESHOLD:
                    is_done+=1

            if is_done==len(self.SCS_ID_list):
                print(status_string)
                break

            iteration+=1

        # Clear syncread parameter storage
        groupSyncRead.clearParam()

In [46]:
servo_1x=sts3032(0, portHandler, packetHandler)
servo_1y=sts3032(1, portHandler, packetHandler)
servo_2x=sts3032(2, portHandler, packetHandler)
servo_2y=sts3032(3, portHandler, packetHandler)

servo_1x.set_speed(3000)
servo_2x.set_speed(3000)
servo_1y.set_speed(3000)
servo_2y.set_speed(3000)

SCServo acc set!
SCServo speed set!
[RxPacketError] Input voltage error!
SCServo acc set!
[RxPacketError] Input voltage error!
SCServo speed set!
SCServo acc set!
SCServo speed set!
[RxPacketError] Input voltage error!
SCServo acc set!
[RxPacketError] Input voltage error!
SCServo speed set!
SCServo speed set!
SCServo speed set!
[RxPacketError] Input voltage error!
SCServo speed set!
[RxPacketError] Input voltage error!
SCServo speed set!


In [47]:
servo_1x.torque_enable()
servo_2x.torque_enable()
servo_1y.torque_enable()
servo_2y.torque_enable()

[RxPacketError] Input voltage error!
[RxPacketError] Input voltage error!


In [54]:
#servo_list=[servo_1x, servo_2x, servo_3x, servo_4x, servo_1y, servo_2y, servo_3y, servo_4y]
servo_list=[servo_1x, servo_2x, servo_1y, servo_2y]
servos=servoset(servo_list, portHandler, packetHandler)
servos.set_zero()

Set Zero Trail  1
SCServo zero set!
Set Zero Trail  1
SCServo zero set!
Set Zero Trail  1
[RxPacketError] Input voltage error!
error [RxPacketError] Input voltage error!
SCServo zero set!
Set Zero Trail  1
[RxPacketError] Input voltage error!
error [RxPacketError] Input voltage error!
SCServo zero set!


In [22]:
goal_position_list  = [7096,7096,6416,7354]
servos.set_angle(goal_position_list)
servos.home()

Start Position: 	[ID:002] Goal:7096 Pres:2048	[ID:004] Goal:7096 Pres:2048	[ID:001] Goal:6416 Pres:2048	[ID:003] Goal:7354 Pres:2048	
Iteration: 0	[ID:002] Goal:7096 Pres:2048	[ID:004] Goal:7096 Pres:2048	[ID:001] Goal:6416 Pres:2048	[ID:003] Goal:7354 Pres:2048	
Iteration: 100	[ID:002] Goal:7096 Pres:3266	[ID:004] Goal:7096 Pres:3270	[ID:001] Goal:6416 Pres:3271	[ID:003] Goal:7354 Pres:3263	
Iteration: 200	[ID:002] Goal:7096 Pres:4875	[ID:004] Goal:7096 Pres:4883	[ID:001] Goal:6416 Pres:4884	[ID:003] Goal:7354 Pres:4873	
Iteration: 300	[ID:002] Goal:7096 Pres:6485	[ID:004] Goal:7096 Pres:6498	[ID:001] Goal:6416 Pres:6350	[ID:003] Goal:7354 Pres:6478	
Iteration: 400	[ID:002] Goal:7096 Pres:7094	[ID:004] Goal:7096 Pres:7093	[ID:001] Goal:6416 Pres:6416	[ID:003] Goal:7354 Pres:7350	
Iteration: 460	[ID:002] Goal:7096 Pres:7096	[ID:004] Goal:7096 Pres:7096	[ID:001] Goal:6416 Pres:6416	[ID:003] Goal:7354 Pres:7354	
Start Position: 	[ID:002] Goal:2048 Pres:7096	[ID:004] Goal:2048 Pres:7096	[

In [21]:
data_cache=[]
for i in range(100):
    data_cache.append(MCP3424_pinhole.convert_and_read())
    if i%100==0:
        print(i)
print('pinhole output:',np.mean(data_cache))

data_cache=[]
for i in range(100):
    data_cache.append(MCP3424_fiber.convert_and_read())
    if i%100==0:
        print(i)
print('fiber output:',np.mean(data_cache))

OSError: [Errno 121] Remote I/O error

In [33]:
with np.load('/home/rydpi5/servomotor/Feetech-Servo-SDK-main/args20240809a.npz') as args_data:
    tl1_x_array=args_data['tl1_x_array']
    tl2_x_array=args_data['tl2_x_array']
    tl3_x_array=args_data['tl3_x_array']
    tl4_x_array=args_data['tl4_x_array']
    array_products=args_data['array_products']
    tl1_y_array=args_data['tl1_y_array']
    tl2_y_array=args_data['tl2_y_array']
    tl3_y_array=args_data['tl3_y_array']
    tl4_y_array=args_data['tl4_y_array']

print(array_products)
print(len(array_products))
print(tl1_x_array)
print(tl2_x_array)
print(tl3_x_array)
print(tl4_x_array)

print(tl1_y_array)
print(tl2_y_array)
print(tl3_y_array)
print(tl4_y_array)

[[ 0.0497      0.05        0.         -0.03075953]
 [ 0.0497      0.05        0.         -0.02752341]
 [ 0.0497      0.05        0.         -0.02428672]
 ...
 [ 0.0517      0.05        0.         -0.02428672]
 [ 0.0517      0.05        0.         -0.02752341]
 [ 0.0517      0.05        0.         -0.03075953]]
400
[-14336 -12611 -10887  -9162  -7437  -5713  -3988  -2264   -539   1186
   2910   4635   6360   8084   9809  11533  13258  14983  16707  18432
  18432  16707  14983  13258  11533   9809   8084   6360   4635   2910
   1186   -539  -2264  -3988  -5713  -7437  -9162 -10887 -12611 -14336
 -14336 -12611 -10887  -9162  -7437  -5713  -3988  -2264   -539   1186
   2910   4635   6360   8084   9809  11533  13258  14983  16707  18432
  18432  16707  14983  13258  11533   9809   8084   6360   4635   2910
   1186   -539  -2264  -3988  -5713  -7437  -9162 -10887 -12611 -14336
 -14336 -12611 -10887  -9162  -7437  -5713  -3988  -2264   -539   1186
   2910   4635   6360   8084   9809  11533  1

In [34]:
data_saved=[]
dy0_saved=[]
dy1_saved=[]
tl1_y_n_saved=[]
tl1_x_n_saved=[]

filename='/home/rydpi5/servomotor/Feetech-Servo-SDK-main/test20240809a.npz'

for i in range(len(tl1_y_array)):
    dy0=array_products[i][0]
    dy1=array_products[i][1]
    tl1_y_n=array_products[i][2]
    tl1_x_n=array_products[i][3]
    #tl1_x_array[i], tl2_x_array[i],
    sync_angle_list=[tl1_x_array[i], tl2_x_array[i], tl1_y_array[i], tl2_y_array[i]]
    servos.set_angle(sync_angle_list)

    data_cache=[]
    for j in range(100):
        data_cache.append(MCP3424_pinhole.convert_and_read())
    data_saved.append(np.mean(data_cache))
    
    print('saving..., data is', np.mean(data_cache))
    dy0_saved.append(dy0)
    dy1_saved.append(dy1)
    tl1_y_n_saved.append(tl1_y_n)
    tl1_x_n_saved.append(tl1_x_n)
    np.savez(filename, data_saved=np.array(data_saved), dy0_saved=np.array(dy0_saved), dy1_saved=np.array(dy1_saved), tl1_y_n_saved=np.array(tl1_y_n_saved), tl1_x_n_saved=np.array(tl1_x_n_saved))

servos.home()

Start Position: 	[ID:005] Goal:-14336 Pres:2048	[ID:008] Goal:18607 Pres:2048	[ID:006] Goal:2048 Pres:2048	[ID:007] Goal:2048 Pres:2048	
Iteration: 0	[ID:005] Goal:-14336 Pres:2048	[ID:008] Goal:18607 Pres:2048	[ID:006] Goal:2048 Pres:2048	[ID:007] Goal:2048 Pres:2048	
Iteration: 100	[ID:005] Goal:-14336 Pres:531	[ID:008] Goal:18607 Pres:3555	[ID:006] Goal:2048 Pres:2048	[ID:007] Goal:2048 Pres:2048	
Iteration: 200	[ID:005] Goal:-14336 Pres:-1883	[ID:008] Goal:18607 Pres:5971	[ID:006] Goal:2048 Pres:2048	[ID:007] Goal:2048 Pres:2048	
Iteration: 300	[ID:005] Goal:-14336 Pres:-4300	[ID:008] Goal:18607 Pres:8387	[ID:006] Goal:2048 Pres:2047	[ID:007] Goal:2048 Pres:2047	
Iteration: 400	[ID:005] Goal:-14336 Pres:-6718	[ID:008] Goal:18607 Pres:10800	[ID:006] Goal:2048 Pres:2048	[ID:007] Goal:2048 Pres:2048	
Iteration: 500	[ID:005] Goal:-14336 Pres:-9132	[ID:008] Goal:18607 Pres:13213	[ID:006] Goal:2048 Pres:2048	[ID:007] Goal:2048 Pres:2049	
Iteration: 600	[ID:005] Goal:-14336 Pres:-11550	[I

In [8]:
servo_1x.torque_disable()
servo_2x.torque_disable()
servo_1y.torque_disable()
servo_2y.torque_disable()

# servo_3x.torque_disable()
# servo_3y.torque_disable()
# servo_4x.torque_disable()
# servo_4y.torque_disable()